# Project 3: Machine Learning on MNIST Dataset
## Thành viên A - MNIST Analysis
**Các Task:** 2.1 - Chuẩn bị Dataset | 2.2 - Decision Tree | 2.3 - Hyperparameter Tuning | 2.4 - Neural Network | 2.5 - Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import graphviz
import os

print("✓ All libraries imported successfully!")
sns.set_style('whitegrid')
os.makedirs('../figures', exist_ok=True)

## TASK 2.1: Chuẩn bị Dataset
- Load MNIST data
- Stratified split 80% train, 20% validation
- Visualize class distributions

In [ ]:
print("=" * 60)
print("TASK 2.1: Preparing Dataset")
print("=" * 60)

# Load MNIST dataset
(X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()

print(f"✓ Loaded MNIST (28x28 pixels)")
print(f"Training samples : {X_train_full.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")
print(f"Image size       : {X_train_full.shape[1]} x {X_train_full.shape[2]}")
print(f"Classes          : {np.unique(y_train_full)}")

# Flatten images from (28,28) -> (784,)
X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

print("\nAfter flatten:")
print("Training data shape:", X_train_full.shape)
print("Testing data shape :", X_test.shape)

In [ ]:
# Split training set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=42,
    shuffle=True
)

# Normalize pixel values
X_train = X_train.astype(np.float32) / 255.0
X_val = X_val.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

print("Dataset Split:")
print(f"Training set   : {X_train.shape}")
print(f"Validation set : {X_val.shape}")
print(f"Test set       : {X_test.shape}")

In [ ]:
# Visualize class distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

datasets = [
    y_train_full,
    y_train,
    y_val,
    y_test
]
titles = ['Original Dataset', 'Training Set (80%)', 'Validation Set (20%)', 'Test Set']
colors = ['steelblue', 'green', 'orange', 'red']

for idx, (ax, data, title, color) in enumerate(zip(axes.flat, datasets, titles, colors)):
    unique, counts = np.unique(data, return_counts=True)
    ax.bar(unique, counts, color=color, alpha=0.7, edgecolor='black')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Class Label')
    ax.set_ylabel('Number of Samples')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticks(range(10))

plt.tight_layout()
plt.savefig('../figures/mnist_01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Class distribution visualization saved!")

## TASK 2.2: Building Decision Tree Classifier

In [ ]:
print("="*60)
print("TASK 2.2: Building Decision Tree Classifier")
print("="*60)

# Initialize and train Decision Tree with entropy (information gain)
dt_classifier = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42,
    max_depth=None
)

print("\n Training Decision Tree...")
dt_classifier.fit(X_train, y_train)

# Evaluate
train_acc = dt_classifier.score(X_train, y_train)
val_acc = dt_classifier.score(X_val, y_val)

print(f"✓ Training completed!")
print(f"  Training Accuracy: {train_acc:.4f}")
print(f"  Validation Accuracy: {val_acc:.4f}")
print(f"  Tree Depth: {dt_classifier.get_depth()}")
print(f"  Leaf Nodes: {dt_classifier.get_n_leaves()}")

In [ ]:
# ------------------------------------------------------------
# Visualize Decision Tree with Graphviz
# ------------------------------------------------------------
print("\nVisualizing Decision Tree (max_depth=4 for readability)...")

dot_data = export_graphviz(
    dt_classifier,
    out_file=None,
    feature_names=[f'pixel_{i}' for i in range(X_train.shape[1])],
    class_names=[str(i) for i in range(10)],
    filled=True,
    rounded=True,
    special_characters=True,
    max_depth=4
)

graph = graphviz.Source(dot_data)

# Hiển thị trực tiếp trong notebook
display(graph)

# Lưu hình để đưa vào báo cáo
os.makedirs("../figures", exist_ok=True)
graph.render(
    filename="../figures/mnist_02_dt_tree",
    format="png",
    cleanup=True
)

print("✓ Decision Tree visualization displayed successfully!")
print("✓ Figure saved to ../figures/mnist_02_dt_tree.png")

## TASK 2.3: Hyperparameter Tuning for Decision Tree

In [ ]:
print("=" * 60)
print("TASK 2.3: Hyperparameter Tuning")
print("=" * 60)

# Hyperparameter search space
param_grid = {
    "max_depth": [5, 10, 15, 20, 25, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

best_score = 0
best_params = None
best_dt = None

results = []

print("\nSearching for the best hyperparameters...")

# Grid Search using validation set
for max_depth in param_grid["max_depth"]:
    for min_split in param_grid["min_samples_split"]:
        for min_leaf in param_grid["min_samples_leaf"]:

            model = DecisionTreeClassifier(
                criterion="entropy",
                max_depth=max_depth,
                min_samples_split=min_split,
                min_samples_leaf=min_leaf,
                random_state=42
            )

            # Train on training set
            model.fit(X_train, y_train)

            # Evaluate on validation set
            val_pred = model.predict(X_val)
            val_acc = accuracy_score(y_val, val_pred)

            results.append({
                "max_depth": max_depth,
                "min_samples_split": min_split,
                "min_samples_leaf": min_leaf,
                "validation_accuracy": val_acc
            })

            if val_acc > best_score:
                best_score = val_acc
                best_params = {
                    "max_depth": max_depth,
                    "min_samples_split": min_split,
                    "min_samples_leaf": min_leaf
                }
                best_dt = model

print("✓ Hyperparameter search completed!")

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Performance of best model
train_acc_dt = best_dt.score(X_train, y_train)
val_acc_tuned = best_dt.score(X_val, y_val)

print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS")
print("=" * 60)

for param, value in best_params.items():
    print(f"{param}: {value}")

print(f"\nTraining Accuracy  : {train_acc_dt:.4f}")
print(f"Validation Accuracy: {val_acc_tuned:.4f}")

# Top 10 parameter combinations
print("\nTop 10 Parameter Combinations:")

display(
    results_df.sort_values(
        by="validation_accuracy",
        ascending=False
    ).head(10)
)

In [ ]:
# ============================================================
# Hyperparameter Tuning Heatmap
# ============================================================

heatmap_df = results_df.copy()

# Replace NaN (None) for visualization only
heatmap_df["max_depth"] = heatmap_df["max_depth"].fillna("None")

# Keep display order
heatmap_df["max_depth"] = pd.Categorical(
    heatmap_df["max_depth"],
    categories=[5, 10, 15, 20, 25, 30, "None"],
    ordered=True
)

pivot = heatmap_df[
    heatmap_df["min_samples_split"] == 2
].pivot_table(
    index="max_depth",
    columns="min_samples_leaf",
    values="validation_accuracy"
)

plt.figure(figsize=(8,5))

sns.heatmap(
    pivot,
    annot=True,
    fmt=".4f",
    cmap="YlOrRd"
)

plt.title("Validation Accuracy by max_depth & min_samples_leaf")
plt.xlabel("min_samples_leaf")
plt.ylabel("max_depth")

plt.tight_layout()

os.makedirs("../figures", exist_ok=True)

plt.savefig(
    "../figures/mnist_23_tuning_heatmap.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("✓ Heatmap saved to ../figures/mnist_23_tuning_heatmap.png")

## TASK 2.4: Building Neural Network Classifier

In [ ]:
print("="*60)
print("TASK 2.4: Building Neural Network (MLP)")
print("="*60)

# Initialize MLP Classifier
mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # 2 hidden layers
    activation='relu',              # Non-linear activation
    solver='adam',                  # Adam optimizer
    learning_rate='adaptive',
    max_iter=500,
    early_stopping=False,
    random_state=42,
    verbose=0
)

print("\nMLP Configuration:")
print(f"  Hidden Layers: (128, 64)")
print(f"  Activation: relu")
print(f"  Solver: adam")
print(f"  Max Iterations: 500")

print("\nTraining MLP...")
mlp_classifier.fit(X_train, y_train)
print("✓ Training completed!")

# Evaluate
train_acc_mlp = mlp_classifier.score(X_train, y_train)
val_acc_mlp = mlp_classifier.score(X_val, y_val)

print(f"\n  Training Accuracy: {train_acc_mlp:.4f}")
print(f"  Validation Accuracy: {val_acc_mlp:.4f}")
print(f"  Iterations: {mlp_classifier.n_iter_}")

## TASK 2.5: Performance Evaluation and Comparison

In [ ]:
print("="*60)
print("TASK 2.5: Test Set Evaluation")
print("="*60)

# Predictions
y_test_pred_dt = best_dt.predict(X_test)
y_test_pred_mlp = mlp_classifier.predict(X_test)

# Accuracy
test_acc_dt = accuracy_score(y_test, y_test_pred_dt)
test_acc_mlp = accuracy_score(y_test, y_test_pred_mlp)

print(f"\nTEST SET ACCURACY:")
print(f"  Decision Tree: {test_acc_dt:.4f}")
print(f"  Neural Network: {test_acc_mlp:.4f}")
print(f"  Difference: {abs(test_acc_mlp - test_acc_dt):.4f}")

In [ ]:
# Classification Report - Decision Tree
print("\n" + "="*60)
print("DECISION TREE - Classification Report")
print("="*60)

class_names = [str(i) for i in range(10)]
print(classification_report(y_test, y_test_pred_dt, target_names=class_names))

In [ ]:
# Classification Report - MLP
print("\n" + "="*60)
print("NEURAL NETWORK - Classification Report")
print("="*60)

print(classification_report(y_test, y_test_pred_mlp, target_names=class_names))

In [ ]:
# Confusion Matrix - Decision Tree
cm_dt = confusion_matrix(y_test, y_test_pred_dt)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Decision Tree', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('../figures/mnist_03_cm_dt.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Confusion Matrix (Decision Tree) saved!")

In [ ]:
# Confusion Matrix - MLP
cm_mlp = confusion_matrix(y_test, y_test_pred_mlp)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Neural Network (MLP)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('../figures/mnist_04_cm_mlp.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Confusion Matrix (MLP) saved!")

In [ ]:
precision_dt, recall_dt, f1_dt, _ = precision_recall_fscore_support(
    y_test, y_test_pred_dt, average=None
)
precision_mlp, recall_mlp, f1_mlp, _ = precision_recall_fscore_support(
    y_test, y_test_pred_mlp, average=None
)

x = np.arange(10)
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, f1_dt, width, label='Decision Tree', color='steelblue', alpha=0.8)
ax.bar(x + width/2, f1_mlp, width, label='MLP', color='darkorange', alpha=0.8)
ax.set_xlabel('Class')
ax.set_ylabel('F1-Score')
ax.set_title('F1-Score per Class: Decision Tree vs MLP')
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in range(10)])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/mnist_05_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ F1 comparison chart saved to ../figures/mnist_05_f1_comparison.png")

## INSIGHTS & DETAILED COMPARISON

In [ ]:
print("\n" + "="*70)
print("DETAILED ANALYSIS & INSIGHTS")
print("="*70)

# 1. Overall accuracy comparison
print("\n1. ACCURACY COMPARISON:")
comparison_data = {
    'Model': ['Decision Tree (Tuned)', 'Neural Network (MLP)'],
    'Training': [train_acc_dt, train_acc_mlp],
    'Validation': [val_acc_tuned, val_acc_mlp],
    'Test': [test_acc_dt, test_acc_mlp] 
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# 2. Per-class F1 scores
print("\n2. PER-CLASS F1-SCORE COMPARISON:")

class_comparison = pd.DataFrame({
    'Class': range(10),
    'DT_F1': np.round(f1_dt, 4),
    'MLP_F1': np.round(f1_mlp, 4),
    'Better': ['MLP' if f1_mlp[i] > f1_dt[i] else 'DT' if f1_dt[i] > f1_mlp[i] else 'Tie' 
               for i in range(10)]
})
print(class_comparison.to_string(index=False))

In [ ]:
# 3. Key insights
print("\n3. KEY INSIGHTS:")
print("="*70)

dt_wins = sum(f1_dt > f1_mlp)
mlp_wins = sum(f1_mlp > f1_dt)

print(f"""
DECISION TREE:
  • Best Parameters: {best_params}
  • Test Accuracy: {test_acc_dt:.4f}
  • Classes where DT is better: {dt_wins}/10
  • Macro F1-Score: {f1_dt.mean():.4f}
  • Training Time: Fast
  • Interpretability: Excellent (white-box)
  • Strengths:
    - Easy to understand and explain
    - No feature scaling needed
    - Fast inference
  • Weaknesses:
    - Prone to overfitting (depth={best_dt.get_depth()})
    - May not capture complex patterns as well as neural networks

NEURAL NETWORK (MLP):
  • Architecture: 2 hidden layers (128, 64 neurons)
  • Test Accuracy: {test_acc_mlp:.4f}
  • Classes where MLP is better: {mlp_wins}/10
  • Macro F1-Score: {f1_mlp.mean():.4f}
  • Training Time: Moderate
  • Interpretability: Limited (black-box)
  • Strengths:
    - Better at capturing non-linear patterns
    - Generally higher accuracy on complex datasets
    - Scalable to deeper architectures
  • Weaknesses:
    - Requires hyperparameter tuning
    - Computationally more expensive than decision trees
    - Less interpretable

WINNER: {'MLP (Neural Network)' if test_acc_mlp > test_acc_dt else 'Decision Tree'}
  Accuracy difference: {abs(test_acc_mlp - test_acc_dt):.4f}
""")

In [ ]:
# 4. Most confused classes
print("\n4. MOST CONFUSED CLASS PAIRS:")
print("="*70)

print("\nDecision Tree Confusions:")
confusions_dt = []
for i in range(10):
    for j in range(i+1, 10):
        total_confusion = cm_dt[i][j] + cm_dt[j][i]
        if total_confusion > 0:
            confusions_dt.append((i, j, total_confusion))

confusions_dt.sort(key=lambda x: x[2], reverse=True)
for digit1, digit2, count in confusions_dt[:5]:
    print(f"   Class {digit1} ↔ Class {digit2}: {count} confusions")

print("\nMLP Confusions:")
confusions_mlp = []
for i in range(10):
    for j in range(i+1, 10):
        total_confusion = cm_mlp[i][j] + cm_mlp[j][i]
        if total_confusion > 0:
            confusions_mlp.append((i, j, total_confusion))

confusions_mlp.sort(key=lambda x: x[2], reverse=True)
for digit1, digit2, count in confusions_mlp[:5]:
    print(f"   Class {digit1} ↔ Class {digit2}: {count} confusions")

In [ ]:
# Final Summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

summary_text = f"""
DATASET INFORMATION:
   • Total Samples: {X_train_full.shape[0] + X_test.shape[0]}
   • Features: {X_train.shape[1]} (28×28 pixel images)
   • Classes: 10 (digits 0-9)
   • Train/Val/Test Split: {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}

MODEL RESULTS:
  Decision Tree (Tuned):
    ✓ Best hyperparameters: {best_params}
    ✓ Test Accuracy: {test_acc_dt:.4f}
    ✓ Average Precision: {precision_dt.mean():.4f}
    ✓ Average Recall: {recall_dt.mean():.4f}
    ✓ Average F1: {f1_dt.mean():.4f}
  
  Neural Network (MLP):
    ✓ Architecture: Input(784) → Hidden(128) → Hidden(64) → Output(10)
    ✓ Test Accuracy: {test_acc_mlp:.4f}
    ✓ Average Precision: {precision_mlp.mean():.4f}
    ✓ Average Recall: {recall_mlp.mean():.4f}
    ✓ Average F1: {f1_mlp.mean():.4f}

RECOMMENDATION:
  {'MLP is recommended - better accuracy and generalization' if test_acc_mlp > test_acc_dt else 'Decision Tree is recommended - simpler and interpretable'}
"""

print(summary_text)
print("\nALL TASKS COMPLETED SUCCESSFULLY!")
print("\nOutput files generated:")
print("  • ../figures/mnist_01_class_distribution.png")
print("  • ../figures/mnist_02_dt_tree.png")
print("  • ../figures/mnist_03_cm_dt.png")
print("  • ../figures/mnist_04_cm_mlp.png")
print("  • ../figures/mnist_05_f1_comparison.png") 
print("  • ../figures/mnist_23_tuning_heatmap.png")  